In [13]:
import sys

sys.path.append('../src')

In [14]:
import pandas as pd

import os.path
from chromatin_model.loaders.packed import load_chromatin_model_ensemble_from_filesystem
from utils.filesystem_utils import get_local_filesystem

In [15]:
gm12878_chr6 = load_chromatin_model_ensemble_from_filesystem(
    fs=get_local_filesystem(),
    data_path=os.path.abspath("../data/haploblocks"),
    model_name="GM12878_full_2_chr6"
)


In [16]:
haploblocks = pd.read_csv("../data/haploblocks/haploblock_boundaries_chr6.tsv", sep="\t")
haploblocks = haploblocks.rename(columns={"START": "start", "END": "end"})
haploblocks["start_model_position"] = (haploblocks["start"] - gm12878_chr6.first_bin) // gm12878_chr6.resolution + 1
haploblocks["end_model_position"] = (haploblocks["end"] - gm12878_chr6.first_bin) // gm12878_chr6.resolution + 1
haploblocks["center_model_position"] = (haploblocks["start_model_position"] + haploblocks["end_model_position"]) // 2
haploblocks["haploblock_id"] = "chr6:" + haploblocks["start"].astype(str) + "-" + haploblocks["end"].astype(str)
haploblocks.set_index("haploblock_id", inplace=True)

# cartesian product of haploblocks on itself
haploblocks = haploblocks.reset_index().merge(
    haploblocks.reset_index(),
    how="cross",
    suffixes=("_a", "_b")
)

# remove self-comparisons and inverted duplicates
haploblocks = haploblocks[
    (haploblocks["haploblock_id_a"] != haploblocks["haploblock_id_b"])
    & (haploblocks["haploblock_id_a"] < haploblocks["haploblock_id_b"])
].reset_index(drop=True)

haploblocks["distance"] = haploblocks[["center_model_position_a", "center_model_position_b"]].apply(
    lambda row: gm12878_chr6.distance_distribution(row["center_model_position_a"], row["center_model_position_b"]),
    axis=1
)

haploblocks["distance_mean"] = haploblocks["distance"].apply(lambda dist: dist.mean())
haploblocks["distance_std"] = haploblocks["distance"].apply(lambda dist: dist.std())

haploblocks = haploblocks[["haploblock_id_a", "start_a", "end_a", "haploblock_id_b", "start_b", "end_b", "distance", "distance_mean", "distance_std"]]
haploblocks

,haploblock_id_a,start_a,end_a,haploblock_id_b,start_b,end_b,distance,distance_mean,distance_std
0,chr6:711055-761032,711055,761032,chr6:761032-761243,761032,761243,"[7.322918, 7.2445316, 7.9951024, 7.5382996, 7....",7.258743,0.856646
1,chr6:711055-761032,711055,761032,chr6:761243-826312,761243,826312,"[6.450018, 5.102061, 5.3873982, 4.987861, 4.05...",5.242896,0.750772
2,chr6:711055-761032,711055,761032,chr6:826312-827388,826312,827388,"[7.5358925, 3.8552415, 5.2460694, 4.453316, 8....",5.999178,2.077318
3,chr6:711055-761032,711055,761032,chr6:827388-840475,827388,840475,"[8.650343, 4.2239575, 5.5943627, 4.4770494, 9....",6.694811,2.449944
4,chr6:711055-761032,711055,761032,chr6:840475-889708,840475,889708,"[14.451623, 8.30593, 9.992073, 7.925118, 14.60...",11.727876,2.946394
...,...,...,...,...,...,...,...,...,...
975101,chr6:169994555-169995100,169994555,169995100,chr6:98490361-98490373,98490361,98490373,"[105.23169, 123.26271, 45.613106, 87.88096, 18...",131.522964,49.195831
975102,chr6:169994555-169995100,169994555,169995100,chr6:98490373-98771082,98490373,98771082,"[103.17177, 120.3575, 43.958294, 88.38988, 185...",129.776245,49.309742
975103,chr6:169994555-169995100,169994555,169995100,chr6:98771082-98771918,98771082,98771918,"[101.4663, 117.74182, 42.66876, 88.8536, 183.9...",128.282318,49.429214
975104,chr6:169994555-169995100,169994555,169995100,chr6:98771918-99225800,98771918,99225800,"[99.5388, 114.418945, 41.3682, 89.461876, 182....",126.491310,49.604507


In [17]:
haploblocks.to_csv("../data/haploblocks/haploblock_distances_chr6.csv", sep=";", index=False)

In [19]:
haploblocks.to_parquet("../data/haploblocks/haploblock_distances_chr6.parquet", index=False)